# 10.9 圖片和描述怎麼比較配對程度？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像把名字卡與照片配對**

對角線是正配對；重複描述可能造成假負例。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/contrastive.svg")))

**先想一想：**相同方向但向量長度2倍，相似度應加倍嗎？

先在獨立實驗把圖與文字各編成向量，正規化後點積就是cosine相似度。矩陣對角線是原來的配對，不必用生成LLM纔有圖文對齊。

**把直覺寫成數學：**similarity=normalize(image)@normalize(text).T；值在−1到1附近。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
image = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
text = torch.tensor([[2.0, 0.0], [0.0, 3.0]])
similarity = F.normalize(image, dim=-1) @ F.normalize(text, dim=-1).T
print(similarity)
assert torch.allclose(similarity, torch.eye(2))

**如何讀結果：**這是手設可查矩陣，不是CLIP訓練成果。每列問：這張圖最像哪段文字？

**只改一件事：**只交換文字兩列，觀察正確配對欄位改變。
